In [ ]:
import cv2
import numpy as np

caminho = "garrafa.jpeg"
img = cv2.imread(caminho)

if img is None:
    raise FileNotFoundError(f"Não foi possível carregar '{caminho}'.")

# 1. Redimensionar mantendo proporção para agilizar e padronizar kernels morfológicos
altura_orig, largura_orig = img.shape[:2]
nova_largura = 600
escala = nova_largura / float(largura_orig)
nova_altura = int(altura_orig * escala)
img_redim = cv2.resize(img, (nova_largura, nova_altura), interpolation=cv2.INTER_AREA)

# 2. Suavização bilateral: remove textura da parede preservando bordas da garrafa
suave = cv2.bilateralFilter(img_redim, d=9, sigmaColor=75, sigmaSpace=75)

# 3. Segmentação cromática no espaço HSV
# A garrafa é rosa: H (tom vermelho/magenta), S (saturação não nula)
hsv = cv2.cvtColor(suave, cv2.COLOR_BGR2HSV)

# Faixa de Rosa/Magenta/Vermelho claro (cobre tanto o final quanto o início da roda de cores)
limiar_inf1 = np.array([140, 30, 80], dtype=np.uint8)
limiar_sup1 = np.array([179, 255, 255], dtype=np.uint8)

limiar_inf2 = np.array([0, 30, 80], dtype=np.uint8)
limiar_sup2 = np.array([15, 255, 255], dtype=np.uint8)

mascara1 = cv2.inRange(hsv, limiar_inf1, limiar_sup1)
mascara2 = cv2.inRange(hsv, limiar_inf2, limiar_sup2)
mascara_cor = cv2.bitwise_or(mascara1, mascara2)

# 4. Morfologia Matemática para selar a garrafa
# 4.1 Fechamento grande: funde o corpo da garrafa e tampa os reflexos internos de luz
kernel_fechamento = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
mascara_fechada = cv2.morphologyEx(mascara_cor, cv2.MORPH_CLOSE, kernel_fechamento, iterations=2)

# 4.2 Abertura: remove pontinhos isolados capturados na parede e na madeira
kernel_abertura = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
mascara_limpa = cv2.morphologyEx(mascara_fechada, cv2.MORPH_OPEN, kernel_abertura, iterations=2)

# 5. Filtragem por Área: Isolar apenas o maior objeto conexo (a Garrafa)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mascara_limpa)

mascara_garrafa = np.zeros_like(mascara_limpa)
if num_labels > 1:
    # Ignora o índice 0 (fundo) e localiza o componente de maior área
    maior_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mascara_garrafa[labels == maior_label] = 255

# Preenchimento de furos restantes na máscara final via dilatação/fechamento
mascara_garrafa = cv2.morphologyEx(mascara_garrafa, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))

# 6. Extração da Fronteira Interna Morfológica: β_interna(A) = A - (A ⊖ B)
kernel_borda = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
erodida = cv2.erode(mascara_garrafa, kernel_borda, iterations=1)
fronteira_interna = cv2.subtract(mascara_garrafa, erodida)

# 7. Sobreposição visual na imagem colorida (contorno verde neon de 2 pixels)
borda_visivel = cv2.dilate(fronteira_interna, kernel_borda, iterations=1)
resultado = img_redim.copy()
resultado[borda_visivel == 255] = [0, 255, 0]

# 8. Salvando os arquivos do pipeline
cv2.imwrite("garrafa_01_mascara_hsv.jpg", mascara_cor)
cv2.imwrite("garrafa_02_mascara_solida.jpg", mascara_garrafa)
cv2.imwrite("garrafa_03_fronteira_interna.jpg", fronteira_interna)
cv2.imwrite("garrafa_04_resultado_final.jpg", resultado)

# Exibição
cv2.imshow("1. Mascara HSV Bruta", mascara_cor)
cv2.imshow("2. Apos Morfologia e Maior Componente", mascara_garrafa)
cv2.imshow("3. Fronteira Interna", fronteira_interna)
cv2.imshow("4. Contorno Sobreposto", resultado)
cv2.waitKey(0)
cv2.destroyAllWindows()